# RFSoC4x2 AWG Basic Usage

This notebook shows the minimal workflow for using the RFSoC4x2 AWG overlay: generate two `int16` waveforms, preview them, load them into the DAC BRAM players, and enable or disable the RF outputs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from firmware import OverlayController
from firmware.signals import sawtooth, sine

In [ ]:
DAC_SAMPLE_RATE = 9.8304e9
BUFFER_LEN = 131072
DAC_PEAK = 0.8 * np.iinfo(np.int16).max

In [ ]:
def to_int16(waveform, peak=DAC_PEAK):
    return np.round(np.clip(np.asarray(waveform) * peak, -peak, peak)).astype(np.int16)


def plot_waveform(waveform, sample_rate, samples=2048, title="Waveform"):
    view = np.asarray(waveform)[:samples]
    time_ns = np.arange(view.size) / sample_rate * 1e9

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time_ns, view)
    ax.set_title(title)
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("DAC code")
    ax.grid(True)
    return ax


def plot_spectrum(waveform, sample_rate, title="Spectrum"):
    data = np.asarray(waveform, dtype=float)
    spectrum = np.fft.rfft(data)
    freqs_mhz = np.fft.rfftfreq(data.size, d=1 / sample_rate) / 1e6
    magnitude_db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-12))

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(freqs_mhz, magnitude_db)
    ax.set_title(title)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.grid(True)
    return ax

In [ ]:
dac0_waveform = to_int16(
    sine(freq_hz=100e6, sample_rate=DAC_SAMPLE_RATE, num_samples=BUFFER_LEN)
)

dac2_waveform = to_int16(
    sawtooth(freq_hz=50e6, sample_rate=DAC_SAMPLE_RATE, num_samples=BUFFER_LEN)
)

dac0_waveform.dtype, dac0_waveform.shape, dac2_waveform.dtype, dac2_waveform.shape

In [ ]:
plot_waveform(dac0_waveform, DAC_SAMPLE_RATE, title="DAC0 waveform");
plot_spectrum(dac0_waveform, DAC_SAMPLE_RATE, title="DAC0 spectrum");
plot_waveform(dac2_waveform, DAC_SAMPLE_RATE, title="DAC2 waveform");
plot_spectrum(dac2_waveform, DAC_SAMPLE_RATE, title="DAC2 spectrum");

In [ ]:
ol = OverlayController()
ol.info()

In [ ]:
ol.dac0.load_waveform(dac0_waveform)
ol.dac2.load_waveform(dac2_waveform)

ol.info()

In [ ]:
ol.dac0.enable()
ol.dac2.enable()

ol.dac0.is_enabled(), ol.dac2.is_enabled()

In [ ]:
ol.dac0.disable()
ol.dac2.disable()

ol.info()